## Task 2: Recursive utility functions

In [1]:
from pathlib import Path
import json


def find_log_files(path: str | Path, extension: str = '.log') -> list[Path]:
    """Recursively find all files with the given extension below path."""
    path = Path(path)
    found: list[Path] = []
    for item in path.iterdir():
        if item.is_dir():
            # Recursive step: search inside the subdirectory
            found.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    # Base case: when a directory has no more subdirectories, the loop ends
    return found


def get_active_registers(node: dict | list | str | int | float | bool | None) -> set[str]:
    """Recursively collect register ids where "active" is true."""
    active: set[str] = set()
    if isinstance(node, dict):
        # A register is a dict with "active": true and an "id"
        if node.get('active') is True and 'id' in node:
            active.add(node['id'])
        # Keep searching deeper in all values of the dict
        for value in node.values():
            active |= get_active_registers(value)
    elif isinstance(node, list):
        # Lists can contain dicts or more lists, so recurse on each item
        for item in node:
            active |= get_active_registers(item)
    # Strings, numbers etc. fall through and return the empty set
    return active


log_files = find_log_files('data')
print(len(log_files))

with open('data/store_config.json', encoding='utf-8') as f:
    config = json.load(f)

active_registers = get_active_registers(config)
print(active_registers)

9
{'POS_03', 'POS_01', 'POS_02'}


## Task 3: Cleaning and analysing data with NumPy

In [2]:
import numpy as np


def remove_outliers(prices: np.ndarray | list[float]) -> np.ndarray:
    """Replace values more than 3 standard deviations from the mean
    with the median of the valid prices."""
    prices = np.asarray(prices, dtype=float).ravel()  # 1D float array
    mean = prices.mean()
    std = prices.std()
    is_outlier = np.abs(prices - mean) > 3 * std      # |x - mean| > 3 sigma
    median = np.median(prices[~is_outlier])           # median of valid prices only
    cleaned = prices.copy()                           # do not change the input
    cleaned[is_outlier] = median
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio: refund / (sales + 1e-6)."""
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    # 1e-6 in the denominator avoids division by zero when an item has no sales
    return refund_count / (sales_count + 1e-6)

### Tests of the Task 3 functions

`remove_outliers` is tested on 13 made-up prices where the last one is a
glitch value of 999.0. It is more than 3 standard deviations from the mean,
so it is replaced by the median of the other prices (3.95), while the normal
prices are unchanged.

`compute_refund_ratio` is tested with sales counts `[100, 50, 0]` and refund
counts `[5, 0, 0]`. The result is `[0.05, 0, 0]`, and the item with zero sales
gives 0 instead of a division error because of the `1e-6` term.

Note that with very few data points, one extreme value also raises the mean
and the standard deviation, so the 3-sigma rule works best on large datasets
like the real transaction data.

In [3]:


test = np.array([4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.0, 3.2, 4.4, 3.9, 4.1, 3.6, 999.0])
print(remove_outliers(test))
print(compute_refund_ratio(np.array([100, 50, 0]), np.array([5, 0, 0])))

[4.5  3.   4.2  3.5  3.8  7.5  4.   3.2  4.4  3.9  4.1  3.6  3.95]
[0.05 0.   0.  ]


## Task 4: Data analysis with pandas

In [4]:
import pandas as pd


def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """Records -> DataFrame with datetime index, sorted by time."""
    df = pd.DataFrame(records)
    df['timestamp'] = pd.to_datetime(df['timestamp'])  # one call for the whole column
    return df.set_index('timestamp').sort_index()


def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Total price per item in 1-hour windows."""
    # Sums all statuses, since the task only asks for total price per hour
    return df.groupby('item')['price'].resample('1h').sum().reset_index()


def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item across status categories."""
    pivot = df.pivot_table(index='item', columns='status', values='price',
                           aggfunc='sum', fill_value=0)
    return pivot.reindex(columns=['SUCCESS', 'REFUND', 'CANCELLED'], fill_value=0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item, using compute_refund_ratio from Task 3."""
    # Count transactions per item and status (not sum of price), because
    # the function takes sales_count and refund_count
    counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    # Make sure all three status columns exist, even if one has no transactions
    counts = counts.reindex(columns=['SUCCESS', 'REFUND', 'CANCELLED'], fill_value=0)
    ratio = compute_refund_ratio(counts['SUCCESS'].to_numpy(),
                                 counts['REFUND'].to_numpy())
    return pd.DataFrame({'item': counts.index, 'refund_ratio': ratio})

In [5]:
fake = [
    {'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 08:45:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'REFUND'},
    {'timestamp': '2026-10-15 09:10:00', 'register_id': 'POS_02', 'item': 'Muffin', 'price': 3.5, 'currency': 'EUR', 'status': 'SUCCESS'},
    {'timestamp': '2026-10-15 09:20:00', 'register_id': 'POS_02', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'},
]
df_test = build_dataframe(fake)
print(hourly_revenue(df_test))
print(status_pivot(df_test))
print(item_refund_ratios(df_test))

     item           timestamp  price
0   Latte 2026-10-15 08:00:00    9.0
1   Latte 2026-10-15 09:00:00    4.5
2  Muffin 2026-10-15 09:00:00    3.5
status  SUCCESS  REFUND  CANCELLED
item                              
Latte       9.0     4.5          0
Muffin      3.5     0.0          0
     item  refund_ratio
0   Latte           0.5
1  Muffin           0.0
